In [14]:
import numpy as np
import tensorflow.keras as tk
from keras import layers
from datetime import datetime

In [15]:
# Define o diretório para salvar os logs do TensorBoard
logdir = 'logs/scalars' + datetime.now().strftime('%y%m%d - %H%M%S')

In [16]:
%reload_ext tensorboard

In [17]:
tensorboard_callback = tk.callbacks.TensorBoard(log_dir = logdir)

In [18]:
num_classes = 10

In [19]:
input_shape = (28,28,1)

In [20]:
mnist = tk.datasets.mnist

In [21]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [22]:
x_train.shape, y_train.shape, x_test.shape, y_test.shape

((60000, 28, 28), (60000,), (10000, 28, 28), (10000,))

In [23]:
x_train = x_train.astype("float32") / 255.
x_train = x_test.astype("float32") / 255.


In [24]:
x_train.shape, x_test.shape

((10000, 28, 28), (10000, 28, 28))

In [25]:
x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

In [26]:
x_train.shape, x_test.shape

((10000, 28, 28, 1), (10000, 28, 28, 1))

In [28]:
y_train = tk.utils.to_categorical(y_train, num_classes)
y_test = tk.utils.to_categorical(y_test, num_classes)

In [29]:
input_shape = (28, 28, 1)  
num_classes = 10  

In [32]:
modelo  = tk.Sequential(
    [
        tk.Input(shape = input_shape),
        layers.Conv2D(32, kernel_size = (3,3), activation = 'relu'),
        layers.MaxPooling2D(pool_size = (2,2)),
        layers.Conv2D(64, kernel_size = (3,3), activation = 'relu'),
        layers.MaxPooling2D(pool_size = (2,2)),
        layers.Flatten(),
        layers.Dense(num_classes, activation = 'softmax')
                    ])

In [33]:
batch_size = 100
epochs = 20

In [39]:
modelo.compile(
    optimizer="adam",
    loss="categorical_crossentropy",  # Alterado de sparse_categorical_crossentropy
    metrics=["accuracy"],
)

In [40]:
# Treina o modelo com os dados de treinamento
training_history = modelo.fit(
    x_train,
    y_train,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    callbacks=[tensorboard_callback]
)

Epoch 1/20
90/90 [==============================] - 2s 20ms/step - loss: 2.3050 - accuracy: 0.1048 - val_loss: 2.3002 - val_accuracy: 0.1070
Epoch 2/20
90/90 [==============================] - 2s 18ms/step - loss: 2.3000 - accuracy: 0.1116 - val_loss: 2.3021 - val_accuracy: 0.1010
Epoch 3/20
90/90 [==============================] - 2s 18ms/step - loss: 2.2969 - accuracy: 0.1171 - val_loss: 2.3003 - val_accuracy: 0.0950
Epoch 4/20
90/90 [==============================] - 2s 17ms/step - loss: 2.2926 - accuracy: 0.1224 - val_loss: 2.3057 - val_accuracy: 0.0990
Epoch 5/20
90/90 [==============================] - 2s 18ms/step - loss: 2.2865 - accuracy: 0.1337 - val_loss: 2.3089 - val_accuracy: 0.1120
Epoch 6/20
90/90 [==============================] - 2s 18ms/step - loss: 2.2784 - accuracy: 0.1417 - val_loss: 2.3104 - val_accuracy: 0.0980
Epoch 7/20
90/90 [==============================] - 2s 18ms/step - loss: 2.2683 - accuracy: 0.1554 - val_loss: 2.3291 - val_accuracy: 0.1020
Epoch 8/20
90

In [41]:
modelo.save('mnist_20_epochs.keras')

In [42]:
# Inicia o TensorBoard para visualização dos logs
get_ipython().run_line_magic('tensorboard', '--logdir logs/scalars --port 6607')

Reusing TensorBoard on port 6607 (pid 10208), started 2:07:36 ago. (Use '!kill 10208' to kill it.)

In [44]:
from sklearn.metrics import confusion_matrix, classification_report

In [45]:
y_preds = modelo.predict(x_test)

313/313 [==============================] - 1s 2ms/step


In [46]:
y_preds_classes = np.argmax(y_preds, axis = 1)
y_real_classes = np.argmax(y_test, axis = 1)

In [47]:
# Calcula a matriz de confusão e a imprime
mat_conf = confusion_matrix(y_real_classes, y_preds_classes)
print("\nMatriz de Confusão \n", mat_conf)


Matriz de Confusão 
 [[ 68 116 137  45  63  91  90 108  93 169]
 [ 67 260  81  73 127 121  85 167  76  78]
 [185 139 101  70  59  60 145  99  67 107]
 [136 212  42  89 129  46  81 133  32 110]
 [ 65 155 142  79  86  81 106 156  23  89]
 [ 87 159  62  51  78  97 119 108  58  73]
 [157 148  93  36  46  56 112 169  68  73]
 [ 80  69 226 116  73  77  41 177  41 128]
 [ 79 130 117 116  58  68  76 159  49 122]
 [160  63 132 145  69 105  63 160  14  98]]
